#### 1. Column Names
#### Display all column names. Identify columns having leading/trailing spaces and
#### rename all columns into a consistent snake_case format.

In [35]:
import pandas as pd
import re

df = pd.read_csv("C:\\Users\\rohit\\Downloads\\matplotlib_messy_sales_2000.csv")

[c for c in df.columns if c != c.strip()]

df.columns = [
    re.sub(r'_+', '_', re.sub(r'[^a-zA-Z0-9]+', '_', c.strip())).strip('_').lower()
    for c in df.columns
]

print("Cleaned columns:", df.columns.tolist())

Cleaned columns: ['customer_id', 'order_date', 'city', 'product_category', 'customer_segment', 'gender', 'sales_channel', 'age', 'units_sold', 'unit_price', 'discount', 'customer_rating', 'is_active', 'returned', 'payment_method', 'gross_sales', 'discount_amount', 'revenue', 'cost', 'profit']


#### 2. Duplicate Records
#### Find the number of duplicate rows. Remove the duplicate records and verify that no  duplicates remain.

In [37]:
df.duplicated().sum()

np.int64(20)

In [44]:
df.drop_duplicates(inplace=True)

In [46]:
df.duplicated().sum()

np.int64(0)

#### 3. Missing Values
#### Display the missing-value count and missing-value percentage for every column.

In [48]:
df.isnull().sum()

customer_id          0
order_date           4
city                25
product_category    24
customer_segment     0
gender               0
sales_channel        0
age                 27
units_sold           4
unit_price          24
discount            28
customer_rating     25
is_active            0
returned             0
payment_method      25
gross_sales          0
discount_amount      0
revenue             25
cost                 0
profit               0
dtype: int64

In [50]:
missing_value = df.isnull().sum() /len(df)*100

missing_value

customer_id         0.00
order_date          0.20
city                1.25
product_category    1.20
customer_segment    0.00
gender              0.00
sales_channel       0.00
age                 1.35
units_sold          0.20
unit_price          1.20
discount            1.40
customer_rating     1.25
is_active           0.00
returned            0.00
payment_method      1.25
gross_sales         0.00
discount_amount     0.00
revenue             1.25
cost                0.00
profit              0.00
dtype: float64

#### 4. Missing Categorical Values
#### Handle missing values in city, product_category, and payment_method using an
#### appropriate strategy.

In [55]:
df[['city', 'product_category', 'payment_method']].isnull().sum()

city                0
product_category    0
payment_method      0
dtype: int64

#### 5. Missing Numerical Values
#### Handle missing values in age, unit_price, discount_pct, customer_rating, and
#### revenue. Explain why you selected mean, median, or another method.

In [71]:
cols = ['age', 'unit_price', 'discount', 'customer_rating', 'revenue']

for col in cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())

df[cols].isnull().sum()

age                0
unit_price         0
discount           0
customer_rating    0
revenue            0
dtype: int64

#### 6. Date Cleaning
#### Convert order_date into a proper datetime format. Identify invalid date values and handle them appropriately.

In [74]:
df = df.copy()

df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')

print("Invalid dates:", df['order_date'].isna().sum())

df = df.dropna(subset=['order_date']).copy()

print(df['order_date'].head())

Invalid dates: 0
0   2025-05-05
1   2025-10-22
2   2025-04-24
3   2025-05-17
4   2025-08-30
Name: order_date, dtype: datetime64[ns]


#### 7. Age Cleaning
#### The age column contains values such as 25 years, 30 yrs, 45Y, and N/A. Extract the
#### numerical age and convert the column into a numeric datatype.

In [75]:
df['age'] = pd.to_numeric(
    df['age'].astype(str).str.extract(r'(\d+)')[0],
    errors='coerce'
)

print(df['age'].head())
print(df['age'].dtype)

0    36
1    38
2    65
3    34
4    41
Name: age, dtype: int64
int64


#### 8. Age Outliers
#### Identify unrealistic ages such as 120, 150, and 200. Detect them using the IQR
#### method and handle them appropriately.

In [76]:
Q1 = df['age'].quantile(0.25)
Q3 = df['age'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

# Identify outliers
outliers = df[(df['age'] < lower) | (df['age'] > upper)]
print("Outliers:", outliers['age'].tolist())

# Replace outliers with median age
median_age = df['age'].median()
df.loc[(df['age'] < lower) | (df['age'] > upper), 'age'] = median_age

print("Outliers handled.")

Outliers: [150, 120, 120, 120, 200, 120, 150, 150, 150, 120, 150, 120, 120]
Outliers handled.


#### 9. Unit Price Cleaning
#### The unit_price column contains values such as ₹15000, $20000, 25000 INR, and 10k.
#### Convert all valid values into a single numeric format.

In [77]:
# Convert unit_price to numeric
def clean_price(x):
    x = str(x).strip().lower()
    
    if 'k' in x:
        return float(x.replace('k', '')) * 1000
    
    x = x.replace('₹', '').replace('$', '').replace('inr', '').replace(',', '').strip()
    return pd.to_numeric(x, errors='coerce')

df['unit_price'] = df['unit_price'].apply(clean_price)

print(df['unit_price'].head())
print(df['unit_price'].dtype)

0    24235.73
1    16789.55
2    24579.97
3     6240.13
4    23491.23
Name: unit_price, dtype: float64
float64


#### 10. Unit Price Outliers
#### Detect extreme unit_price values using the IQR method. Compare the number of
#### outliers before and after treatment.

In [78]:
Q1 = df['unit_price'].quantile(0.25)
Q3 = df['unit_price'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers_before = ((df['unit_price'] < lower) | 
                   (df['unit_price'] > upper)).sum()

median_price = df['unit_price'].median()
df.loc[(df['unit_price'] < lower) | 
       (df['unit_price'] > upper), 'unit_price'] = median_price

outliers_after = ((df['unit_price'] < lower) | 
                  (df['unit_price'] > upper)).sum()

print("Outliers before:", outliers_before)
print("Outliers after:", outliers_after)

Outliers before: 14
Outliers after: 0
